In [ ]:
import os
import datetime
from functools import partial

import colormaps
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from tqdm import tqdm
from jetutils.anyspell import get_persistent_jet_spells, mask_from_spells_pl, subset_around_onset, get_persistent_spell_times_from_som, get_spells, extend_spells, gb_index, make_daily
from jetutils.clustering import Experiment, RAW_REALSPACE, labels_to_centers
from jetutils.data import standardize, extract
from jetutils.definitions import DATADIR, KAPPA
from jetutils.jet_finding import gaussian_smooth_func, do_everything, is_polar_gmix
import polars.selectors as cs

os.environ["RUST_BACKTRACE"] = "1"

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from jetutils.jet_finding import to_one_large, track_jets, pers_from_cross, average_jet_categories
from jetutils.plots import plot_seasonal

for run in ["historical", "ssp370"]:
    path = Path(DATADIR, "CESM2/high_wind", run, "results/1")
    newpath = Path(DATADIR, "CESM2/high_wind", run, "results/2")
    newpath.mkdir(exist_ok=True)
    jets = pl.read_parquet(path.joinpath("jets_fixed.parquet"))
    jets = jets.drop("diff", "is_polar_old")
    jets.write_parquet(newpath.joinpath("jets.parquet"))
    props = pl.read_parquet(path.joinpath("props.parquet"))
    newcat = jets.group_by("member", "time", "jet ID").agg(pl.col("is_polar").mean())
    props = props.drop("is_polar").join(newcat, on=("member", "time", "jet ID"))
    props.write_parquet(newpath.joinpath("props.parquet"))
    phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
    cross = track_jets(phat_jets)
    cross.write_parquet(newpath.joinpath("cross.parquet"))
    pers = pers_from_cross(cross)    
    phat_filter = (pl.col("is_polar") < 0.5) | ((pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8))
    phat_props = props.filter(phat_filter)
    phat_props = average_jet_categories(phat_props, polar_cutoff=0.5)


In [ ]:
_ = plot_seasonal(phat_props, ["mean_lat", "mean_s", "mean_theta"], ncols=2, nrows=2, clear=False)

In [ ]:
props.drop("is_polar").join(newcat, on=("member", "time", "jet ID"))

In [ ]:
new_is_polar = jets_newcat.group_by("member", "time", "jet ID").agg(pl.col("is_polar").mean())
props_ = props.rename({"is_polar": "is_polar_old"}).join(new_is_polar, on=["member", "time", "jet ID"])

In [ ]:
from jetutils.jet_finding import to_one_large
centers = to_one_large(jets_newcat, int_EDJ_threshold=0).group_by(pl.col("time").dt.week(), "jet").agg(pl.col("theta").mean(), pl.col("s").mean()).sort("time", "jet")

In [ ]:
from jetutils.plots import plot_seasonal
from jetutils.jet_finding import average_jet_categories
_ = plot_seasonal(average_jet_categories(props_), ["mean_s", "mean_theta"], ncols=2, nrows=1, clear=False)

In [ ]:
from jetutils.plots import COLORS, COLORS_EXT
from jetutils.definitions import PRETTIER_VARNAME
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv
from jetutils.jet_finding import extract_features
from matplotlib.colors import LinearSegmentedColormap
xys = []
xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
gridsize = 18
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(jets_newcat, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=gridsize)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=gridsize
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.pow(x, 2)
    colors[:, 1] = min_s + f(scaling ** 1.5) * (max_s - min_s)
    colors[:, 2] = max_v - f(scaling) * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * (scaling), 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    # if month > 8:
    #     label = PRETTIER_VARNAME.get(pair[0], pair[0])
    #     unit = UNITS.get(pair[0], "$~$")
    #     ax.set_xlabel(f"{label} [{unit}]")
    # if month % 4 == 1:
    #     label = PRETTIER_VARNAME.get(pair[1], pair[1])
    #     unit = UNITS.get(pair[1], "$~$")
    #     ax.set_ylabel(f"{label} [{unit}]")
    # if pair[0] in ["lev", "theta"]:
    #     ax.invert_xaxis()
    # elif pair[1] in ["lev", "theta"]:
    #     ax.invert_yaxis()

    # ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
    # break
# fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

In [ ]:
import xarray as xr
xr.open_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/full.zarr").to

In [ ]:
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options=dict(token='anon'),
)
vars = ["v_component_of_wind", "u_component_of_wind", "vertical_velocity", "temperature"]
levs = [175, 200, 225, 250, 300, 350]
time_filter = (
    np.isin(ds.time.dt.year, np.arange(1959, 2025))
    & (ds.time.dt.hour % 6 == 0)
)
ds = standardize(ds.isel(time=time_filter)[vars]).sel(lev=levs).sel(lon=np.arange(-80, 40.5, 0.5), lat=np.arange(15, 80.5, 0.5))

In [ ]:
to_comp = ds.drop_encoding().to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/full.zarr", consolidated=False, compute=False, mode="w")

In [ ]:
from dask.diagnostics.progress import ProgressBar
with ProgressBar():
    to_comp.compute()

# rest

In [ ]:
kwargs = dict(
    n_coarsen=1,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)

both_jets = {}
both_paths = {}
for run in ["historical", "ssp370"]:
    path = Path(DATADIR, "CESM2/high_wind", run)
    ds = xr.open_dataset(path.joinpath("ds.zarr"))
    ds = standardize(ds)
    ds["theta"] = ds["t"] * (1000 / ds["lev"]) ** KAPPA
    ds = extract(
        ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
    )
    jets, ph_jets, props, props_full = do_everything(ds, path.joinpath("results/1"), feature_names=("s", "theta"), n_init=3, **kwargs)
    both_paths[run] = path
    both_jets[run] = ph_jets